In [ ]:
import json
import pandas as pd
import os
from datetime import datetime, timedelta


# Load JSON
json_path = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests.json"
with open(json_path, "r") as f:
    contests = json.load(f)

df_contests = pd.DataFrame(contests)


Now we identify the missing metadata 


In [8]:
def missing_meta(meta):
    # Return True if metadata is missing (None or not a dictionary)
    return not isinstance(meta, dict)

# Filter the DataFrame for rows with missing metadata
missing_df = df_contests[df_contests["metadata"].apply(missing_meta)]

# Show results
print(f"Contests missing metadata: {len(missing_df)}\n")



Contests missing metadata: 144



In [14]:

from pathlib import Path

def compute_votes_per_csv(csv_folder):
    """
    Go through every CSV file in a folder, compute the sum of the 'votes' column,
    and return a dictionary {filename: total_votes}.
    """
    folder = Path(csv_folder)
    results = {}

    for csv_file in folder.glob("*.csv"):
        try:
            df = pd.read_csv(csv_file)
            if "votes" in df.columns:
                total_votes = pd.to_numeric(df["votes"], errors="coerce").sum()
                results[csv_file.name] = int(total_votes)
            else:
                results[csv_file.name] = None  # No 'votes' column found
        except Exception as e:
            results[csv_file.name] = f"Error: {e}"

    return results


In [15]:
csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"
vote_sums = compute_votes_per_csv(csv_folder)
print(vote_sums)



{'545.csv': 344095, '551.csv': 583185, '579.csv': 480412, '586.csv': 480072, '592.csv': 1021316, '747.csv': 874526, '753.csv': 1116808, '784.csv': 1225039, '790.csv': 848102, '633.csv': 844596, '627.csv': 1162999, '828.csv': 594974, '814.csv': 565167, '800.csv': 760341, '801.csv': 562261, '815.csv': 534542, '829.csv': 699465, '626.csv': 994702, '632.csv': 663855, '791.csv': 910349, '785.csv': 425704, '752.csv': 1141292, '746.csv': 1124620, '593.csv': 619876, '587.csv': 488546, '578.csv': 593761, '550.csv': 463593, '544.csv': 198246, '552.csv': 519548, '546.csv': 333116, '591.csv': 683358, '585.csv': 356787, '750.csv': 797912, '744.csv': 1110562, '778.csv': 743889, '793.csv': 710753, '787.csv': 683875, '624.csv': 712054, '630.csv': 1159733, '618.csv': 1376577, '803.csv': 810305, '817.csv': 564661, '816.csv': 845170, '802.csv': 724844, '619.csv': 867527, '631.csv': 1065763, '625.csv': 1326529, '786.csv': 1076445, '792.csv': 782090, '779.csv': 701175, '745.csv': 997609, '751.csv': 789736,

In [34]:
import os
import json
import pandas as pd
from pathlib import Path

def check_votes_against_json(json_df, csv_folder):
    """
    For each CSV file in csv_folder:
      - compute sum of 'votes' column
      - compare with num_votes from json_df (if available)
    """
    folder = Path(csv_folder)
    results = []

    for csv_path in folder.glob("*.csv"):
        contest_id = csv_path.stem  # e.g. "522"
        try:
            df = pd.read_csv(csv_path)
            total_votes = pd.to_numeric(df["votes"], errors="coerce").sum()

            # try to find matching contest in JSON
            row = json_df[json_df["contest_id"].astype(str) == str(contest_id)]
            if not row.empty:
                json_votes = row["metadata"].iloc[0].get("num_votes", None)
            else:
                json_votes = None

            results.append({
                "contest_id": contest_id,
                "csv_votes": int(total_votes),
                "json_votes": json_votes,
                "match": json_votes == int(total_votes) if json_votes is not None else None
            })
        except Exception as e:
            results.append({
                "contest_id": contest_id,
                "error": str(e)
            })

    df_results = pd.DataFrame(results)
    print(df_results)
    return df_results


In [35]:
# Load your JSON
json_path = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests.json"
with open(json_path, "r") as f:
    contests = json.load(f)

df_contests = pd.DataFrame(contests)

# Run the check
csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"
df_results = check_votes_against_json(df_contests, csv_folder)


    contest_id  csv_votes  json_votes match  \
0          545   344095.0    344095.0  True   
1          551   583185.0    583185.0  True   
2          579   480412.0    480412.0  True   
3          586   480072.0    480072.0  True   
4          592  1021316.0   1021316.0  True   
..         ...        ...         ...   ...   
379        776        NaN         NaN   NaN   
380        762   901611.0    901611.0  True   
381        574   410803.0    410803.0  True   
382        560   418917.0    418917.0  True   
383        548   501425.0    501425.0  True   

                                        error  
0                                         NaN  
1                                         NaN  
2                                         NaN  
3                                         NaN  
4                                         NaN  
..                                        ...  
379  'NoneType' object has no attribute 'get'  
380                                       NaN  
381

In [23]:
import pandas as pd
from pathlib import Path

def check_votes_against_json(json_df, csv_folder):
    """
    For each contest in the json_file that has metadata:
      - find its corresponding CSV 
      - compute sum of 'votes' column
      - compare to 'num_votes' in metadata

    Returns a DataFrame with results with true when the values match
    """
    folder = Path(csv_folder)
    results = []

    # Filter JSON to keep only contests with non-null metadata
    df_with_meta = json_df[json_df["metadata"].notnull()].copy()

    for _, row in df_with_meta.iterrows():
        contest_id = str(row["contest_id"])
        metadata = row["metadata"]

        # Skip if metadata is not a dict (some entries might be NaN or strings)
        if not isinstance(metadata, dict):
            continue

        csv_path = folder / f"{contest_id}.csv"
        if not csv_path.exists():
            results.append({
                "contest_id": contest_id,
                "csv_found": False,
                "csv_votes": None,
                "json_votes": metadata.get("num_votes"),
                "match": None
            })
            continue

        try:
            df = pd.read_csv(csv_path)
            if "votes" not in df.columns:
                results.append({
                    "contest_id": contest_id,
                    "csv_found": True,
                    "csv_votes": None,
                    "json_votes": metadata.get("num_votes"),
                    "match": None,
                    "error": "No 'votes' column"
                })
                continue

            # Compute total votes
            total_votes = pd.to_numeric(df["votes"], errors="coerce").sum()
            json_votes = metadata.get("num_votes")

            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "csv_votes": int(total_votes),
                "json_votes": json_votes,
                "match": int(total_votes) == json_votes if json_votes is not None else None
            })

        except Exception as e:
            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "error": str(e)
            })

    df_results = pd.DataFrame(results)
    return df_results
print(df_results)

    contest_id  csv_found  csv_votes  json_votes  match
0          510       True      41185       41185   True
1          511       True      28205       28205   True
2          512       True      21574       21574   True
3          513       True      16894       16894   True
4          514       True      95790       95790   True
..         ...        ...        ...         ...    ...
235        759       True     779033      779033   True
236        760       True     807956      807956   True
237        761       True     926127      926127   True
238        762       True     901611      901611   True
239        763       True     889556      889556   True

[240 rows x 5 columns]


In [21]:
json_path = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/contests.json"
csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"

with open(json_path, "r") as f:
    contests = json.load(f)

df_contests = pd.DataFrame(contests)

df_results = check_votes_against_json(df_contests, csv_folder)
print(df_results)
# Filter rows where match == False
mismatches = df_results[df_results["match"] == False]
print("❌ Contests with mismatched vote counts:")
print(mismatches)



    contest_id  csv_found  csv_votes  json_votes  match
0          510       True      41185       41185   True
1          511       True      28205       28205   True
2          512       True      21574       21574   True
3          513       True      16894       16894   True
4          514       True      95790       95790   True
..         ...        ...        ...         ...    ...
235        759       True     779033      779033   True
236        760       True     807956      807956   True
237        761       True     926127      926127   True
238        762       True     901611      901611   True
239        763       True     889556      889556   True

[240 rows x 5 columns]
❌ Contests with mismatched vote counts:
Empty DataFrame
Columns: [contest_id, csv_found, csv_votes, json_votes, match]
Index: []


Now let's add the missing metadata

In [28]:

csv_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"

def update_metadata_from_csv_1(df, base_path=csv_folder):
    for i, row in df.iterrows():
        # Ensure metadata is a dict
        meta = row.get("metadata")
        if not isinstance(meta, dict):
            meta = {}

        # Skip if both fields already exist
        need_caps = ("num_captions" not in meta) 
        need_votes = ("num_votes" not in meta) 
        if not (need_caps or need_votes):
            df.at[i, "metadata"] = meta
            continue

        # Build CSV path from the 'data' field
        data_field = row.get("data")
        if not isinstance(data_field, str) :
            df.at[i, "metadata"] = meta
            continue

        filename = os.path.basename(data_field)      
        csv_path = os.path.join(base_path, filename) 
        # If file not found, skip it
        if not os.path.exists(csv_path):
            df.at[i, "metadata"] = meta
            continue
        # Read CSV and compute values
        try:
            cdf = pd.read_csv(csv_path)
            meta["num_captions"] = int(len(cdf))
            meta["num_votes"] = int(cdf["votes"].sum(skipna=True))
        except Exception as e:
            pass  
        # Write metadata dict back into the DataFrame
        df.at[i, "metadata"] = meta
    return df


# Example usage:
df_contests_1 = update_metadata_from_csv_1(df_contests)


In [43]:
import pandas as pd
from pathlib import Path

def check_votes_all_contests(json_df, csv_folder):
    """
    Goes through all contests in json_df ,
    computes total votes from corresponding CSVs, and compares with num_votes
    in metadata (if available).
    
    Returns a DataFrame with the results.
    """
    folder = Path(csv_folder)
    results = []

    for _, row in json_df.iterrows():
        contest_id = str(row["contest_id"])
        metadata = row.get("metadata", {})

        # ensure metadata is a dict (in case it's NaN or string)
        if not isinstance(metadata, dict):
            metadata = {}

        csv_path = folder / f"{contest_id}.csv"
        try:
            df = pd.read_csv(csv_path)

            # compute total votes
            total_votes = pd.to_numeric(df["votes"], errors="coerce").sum()
            json_votes = metadata.get("num_votes", None)

            # compare (only if json_votes exists)
            match = (json_votes is not None) and (int(total_votes) == int(json_votes))

            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "csv_votes": int(total_votes),
                "json_votes": json_votes,
                "match": match
            })

        except Exception as e:
            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "csv_votes": None,
                "json_votes": metadata.get("num_votes", None),
                "match": None,
                "error": str(e)
            })

    df_results = pd.DataFrame(results)

    # summary print
    total = len(df_results)
    matches = df_results["match"].sum(skipna=True)
    print(f"Total contests checked: {total}")
    print(f"Matching vote counts: {int(matches)}")
    print(f"Mismatches or missing: {total - int(matches)}")

    return df_results


In [45]:
df_results = check_votes_all_contests(df_contests_1, csv_folder)


Total contests checked: 384
Matching vote counts: 384
Mismatches or missing: 0


## Adding missing dates

In [32]:
def add_contest_dates(df):
    """
    Adds contest_day_publish and contest_day_votes to each row's metadata.
    Uses contest 510 as base (22 Feb 2016).
    """
    contest_510_publish = datetime.strptime('2016.02.22', "%Y.%m.%d")
    contest_510_votes = datetime.strptime('2016.02.29', "%Y.%m.%d")

    for i in range(len(df)):
        contest_id = df['contest_id'].iloc[i]
        diff = int(contest_id) - 510
        contest_day_publish = contest_510_publish + timedelta(days=diff * 7)

        # your original assignments
        df['metadata'].iloc[i]['contest_day_publish'] = contest_day_publish
        df['metadata'].iloc[i]['contest_day_votes'] = contest_day_publish + timedelta(days=7)

    return df



#df_contests_1.iloc[250:300]